In [ ]:
with open("../datasets/ali/kg.txt") as f:
    for _ in range(5):
        print(repr(f.readline()))

In [ ]:
from pykeen.triples import TriplesFactory

tf_ali = TriplesFactory.from_path(
    "../datasets/ali/kg.txt",
    load_triples_kwargs={
       "delimiter": " ", 
    }
)

train_tf_ali, val_tf_ali, test_tf_ali = tf_ali.split(
    [0.8, 0.1, 0.10],
)

tf_ali.num_entities, tf_ali.num_relations,train_tf_ali.num_triples, test_tf_ali.num_triples

In [ ]:
tf_lastfm = TriplesFactory.from_path(
    "../datasets/lastfm/kg.txt",
    load_triples_kwargs={
       "delimiter": " ", 
    }
)

train_tf_lastfm, val_tf_lastfm, test_tf_lastfm = tf_lastfm.split(
    [0.80, 0.1, 0.10],
)

tf_lastfm.num_entities, tf_lastfm.num_relations, train_tf_lastfm.num_triples, test_tf_lastfm.num_triples

In [ ]:
tf_mind = TriplesFactory.from_path(
     "../datasets/mind/kg.txt",
     load_triples_kwargs={
        "delimiter": " ",
     }
 )
 
train_tf_mind, val_tf_mind, test_tf_mind = tf_mind.split(
    [0.80, 0.1, 0.10],
)

tf_mind.num_entities, tf_mind.num_relations, train_tf_mind.num_triples, test_tf_mind.num_triples

In [ ]:
train_tf = [train_tf_ali, train_tf_lastfm, train_tf_mind]
val_tf = [val_tf_ali, val_tf_lastfm, val_tf_mind]
test_tf = [test_tf_ali, test_tf_lastfm, test_tf_mind]
ds = ['ali', 'lastfm', 'mind']

In [ ]:
from pykeen.pipeline import pipeline
import torch

for train_tf, test_tf, dataset in zip(val_tf, test_tf, ds):
    result = pipeline(
        training=train_tf,
        testing=test_tf,
        model="RotatE",
        training_kwargs={
            "num_epochs": 1,
        },
        model_kwargs={
            "embedding_dim": 4096,
        }
    )

    kge_model = result.model
    entity_embeddings = kge_model.entity_representations[0](indices=None)
    print(dataset, entity_embeddings.shape)

    torch.save(
        entity_embeddings,
        f"../datasets/{dataset}/embeddings.pt",
    )
    # Save the full model (entity + relation embeddings + interaction) and the
    # split triples (shared ID space) so notebooks/kg_eval.ipynb can run
    # link-prediction metrics (GMR/HMR/AMR/Hits@10) on the exact trained model.
    torch.save(kge_model, f"../datasets/{dataset}/model.pt")
    torch.save(train_tf.mapped_triples, f"../datasets/{dataset}/train_triples.pt")
    torch.save(test_tf.mapped_triples, f"../datasets/{dataset}/test_triples.pt")
